In [ ]:

# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
from statsbombpy import sb  # noqa: F401

from pipeline import get_complete_carries, get_teamseason_matchevents


In [ ]:
arsenal_1516_events = get_teamseason_matchevents(2, 27, 1)

In [ ]:
ff_events = arsenal_1516_events['50_50'].dropna()

3754217

In [ ]:
g1_matchevents = arsenal_1516_events[arsenal_1516_events["match_id"] == 3754217]

In [ ]:
g1_altered = get_complete_carries(g1_matchevents)

In [ ]:
g1_altered = g1_altered.sort_values(by = 'match_seconds').reset_index(drop = True)

In [ ]:
carry_events = g1_matchevents[g1_matchevents['type'].isin(['Carry'])].dropna(axis = 1, how = 'all')

In [ ]:
carry_events = carry_events.sort_values(by = ['match_seconds'])

In [ ]:
dribble_events = g1_matchevents[g1_matchevents['type'].isin(['Dribble'])].dropna(axis = 1, how = 'all')

In [ ]:
dribble_events = dribble_events.sort_values(by = ['match_seconds'])

In [ ]:
carry_events['carry_end_seconds'] = carry_events['match_seconds'] + carry_events['duration']

In [ ]:
carry_events[['x', 'y']] = pd.DataFrame(carry_events['location'].tolist(), index=carry_events.index)

In [ ]:
carry_events[['end_x', 'end_y']] = pd.DataFrame(carry_events['carry_end_location'].tolist(), index=carry_events.index)

In [ ]:
carry_events['carry_distance'] = np.sqrt(
    ((carry_events['x'] - carry_events['end_x'])**2) +
    ((carry_events['y'] - carry_events['end_y'])**2)
)

In [ ]:
next_carry_events = carry_events.shift(-1).copy()

In [ ]:
time_diff = next_carry_events['match_seconds'] - carry_events['carry_end_seconds']
spatial_diff = np.sqrt(
    ((carry_events['end_x'] - next_carry_events['x'])**2) +
    ((carry_events['end_y'] - next_carry_events['y'])**2)
)
player_diff = (carry_events['player_id'] == next_carry_events['player_id'])

In [ ]:
chain_continues = (time_diff <= 0.150) & (spatial_diff <= 0.5) & (player_diff)
chain_starts = ~chain_continues.shift(1, fill_value=False)
carry_events['chain_id'] = chain_starts.cumsum()

In [ ]:
carry_events['total_carries'] = 1

In [ ]:

def my_union_related_events(series):
    ret_set = set()
    for x in series:
        if isinstance(x, list):
            cur_set = set(x)
        elif pd.isna(x):
            cur_set = set()
        else:
            cur_set = set(x)
        ret_set = ret_set.union(cur_set)
    return list(ret_set)


In [ ]:
agg_rules = {col: 'first' for col in carry_events.columns}

agg_rules.update({
    'end_x': 'last',
    'end_y': 'last',
    'carry_end_location': 'last',
    'carry_distance': 'sum',
    'duration': 'sum',
    'total_carries': 'sum',
    'under_pressure': lambda p: p.fillna(False).astype(bool).any(),
    'id': lambda _: np.nan,
    'index': lambda _: np.nan,
    'related_events': my_union_related_events,
})

In [ ]:
complete_carries = carry_events.groupby('chain_id', as_index=False).agg(agg_rules)
complete_carries['type'] = 'Complete Carry'

In [ ]:
g1_matchevents_updated = pd.concat([g1_matchevents, complete_carries], ignore_index = True)
g1_matchevents_updated = g1_matchevents_updated.sort_values(by = 'match_seconds').reset_index(drop = True)

get_complete_carries works